# Unsupervised Learning — K-Means and PCATwo foundational unsupervised techniques applied to **UCI Wine Quality (red)**:- **PCA** projects high-dimensional data onto directions of greatest variance — useful for visualization and dimensionality reduction- **K-Means** finds `k` cluster centroids by alternating between assignment and update steps**Dataset:** 1599 red wine samples, 11 physicochemical features (acidity, sugar, pH, alcohol, etc.). Source: [UCI](https://archive.ics.uci.edu/dataset/186/wine+quality).

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_score, adjusted_rand_scoresns.set_style("whitegrid")np.random.seed(42)

## 1. Load data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv",    "wine_quality_red.csv",)df = pd.read_csv(path)print("Shape:", df.shape)df.head()

## 2. Scale features (essential for both PCA and k-means)

In [ ]:
feat_cols = [c for c in df.columns if c != "quality"]X = df[feat_cols].valuesy = df["quality"].values  # ground-truth quality (we'll only peek at this at the end)X_s = StandardScaler().fit_transform(X)

## 3. PCA → 2D projection

In [ ]:
pca = PCA(n_components=2)X_pca = pca.fit_transform(X_s)print("Variance explained per component:", pca.explained_variance_ratio_)print("Cumulative:", pca.explained_variance_ratio_.cumsum())plt.figure(figsize=(7, 5))sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis",                 alpha=0.6, edgecolor="k", linewidth=0.3)plt.colorbar(sc, label="wine quality (3-8)")plt.xlabel("PC1"); plt.ylabel("PC2")plt.title(f"Wine in PCA space ({pca.explained_variance_ratio_.sum():.1%} variance)")plt.tight_layout(); plt.show()

## 4. Choose k via the elbow method

In [ ]:
inertias = []ks = range(1, 11)for k in ks:    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_s)    inertias.append(km.inertia_)plt.plot(list(ks), inertias, marker="o")plt.xlabel("k"); plt.ylabel("Inertia (within-cluster SSE)")plt.title("Elbow Method")plt.tight_layout(); plt.show()

The elbow appears around k=3 or k=4 — the wines naturally cluster into a few groups.

## 5. Fit k-means with k=4

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_s)clusters = km.labels_print(f"Silhouette score: {silhouette_score(X_s, clusters):.3f}")print(f"ARI vs quality:   {adjusted_rand_score(y, clusters):.3f}")

Note: ARI is low because *quality* (3-8) does not actually cluster cleanly in feature space — wine quality is more nuanced than physicochemistry alone. K-means is finding *real* structure, just not aligned with the quality scores.

## 6. Visualize clusters in PCA space

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))ax[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis",              alpha=0.6, edgecolor="k", linewidth=0.3)ax[0].set_title("Coloured by quality (true label)")ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2")ax[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap="tab10",              alpha=0.6, edgecolor="k", linewidth=0.3)centers_pca = pca.transform(km.cluster_centers_)ax[1].scatter(centers_pca[:, 0], centers_pca[:, 1], s=200,              marker="X", c="red", edgecolor="k", label="centers")ax[1].set_title("Coloured by k-means cluster")ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")ax[1].legend()plt.tight_layout(); plt.show()

## Takeaways- The first 2 principal components capture ~50% of the variance — wine quality data is genuinely high-dimensional.- **K-means** finds 4 reasonably distinct clusters in feature space without any supervision.- These clusters do NOT correspond directly to quality scores (low ARI). They likely correspond to *style* differences (sweet vs dry, high vs low alcohol) that cut across quality grades.- **PCA + k-means together** is a classic exploratory data-analysis pipeline: reduce dimensionality, visualize, hypothesize structure, cluster, interpret.